# Deep Learning Systems

Author: Sukh Sandhu

This notebook trains a baseline CNN and exactly one experimental CNN configuration on the public scikit-learn digits image dataset.

In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
DATA_PATH = Path('data/sklearn_digits.csv')
FIG_DIR = Path('figures')
OUT_DIR = Path('outputs')
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)


## Data Handling

In [2]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=['target']).values.astype('float32') / 16.0
y = df['target'].values.astype('int64')
X = X.reshape(-1, 1, 8, 8)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=42, stratify=y)
train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
test_tensor = torch.tensor(X_test)
print('Dataset:', df.shape, 'Train:', X_train.shape, 'Test:', X_test.shape)

fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X[i, 0], cmap='gray')
    ax.set_title(str(y[i]))
    ax.axis('off')
fig.suptitle('Figure 1. Sample digit images')
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure_1_digit_samples.png', dpi=160)
plt.show()


Dataset: (1797, 65) Train: (1401, 1, 8, 8) Test: (396, 1, 8, 8)


C:\Users\SukhPc\AppData\Local\Temp\ipykernel_56124\2765894937.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model Definition

In [3]:
class DigitCNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(16 * 4 * 4, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def train_model(dropout):
    model = DigitCNN(dropout=dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    history = []
    for epoch in range(8):
        model.train()
        losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            logits = model(test_tensor)
            preds = logits.argmax(dim=1).numpy()
            acc = accuracy_score(y_test, preds)
        history.append({'epoch': epoch + 1, 'loss': float(np.mean(losses)), 'accuracy': float(acc)})
    return model, history, preds


## Baseline and Controlled Experiment

In [4]:
baseline_model, baseline_history, baseline_preds = train_model(dropout=0.0)
experimental_model, experimental_history, experimental_preds = train_model(dropout=0.25)
pd.DataFrame({'baseline_accuracy': [baseline_history[-1]['accuracy']], 'dropout_accuracy': [experimental_history[-1]['accuracy']]})


,baseline_accuracy,dropout_accuracy
0,0.977273,0.972222


## Evaluation

In [5]:
history_df = pd.DataFrame({
    'epoch': [h['epoch'] for h in baseline_history],
    'baseline_accuracy': [h['accuracy'] for h in baseline_history],
    'dropout_accuracy': [h['accuracy'] for h in experimental_history],
    'baseline_loss': [h['loss'] for h in baseline_history],
    'dropout_loss': [h['loss'] for h in experimental_history],
})
display(history_df.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df['epoch'], history_df['baseline_accuracy'], marker='o', label='Baseline CNN')
ax.plot(history_df['epoch'], history_df['dropout_accuracy'], marker='o', label='Dropout CNN')
ax.set_title('Figure 2. Test accuracy by epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure_2_accuracy_curve.png', dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, experimental_preds, ax=ax, colorbar=False)
ax.set_title('Figure 3. Experimental CNN confusion matrix')
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure_3_dropout_confusion_matrix.png', dpi=160)
plt.show()


,epoch,baseline_accuracy,dropout_accuracy,baseline_loss,dropout_loss
0,1,0.8586,0.7904,1.7359,1.4973
1,2,0.9116,0.9293,0.4012,0.4328
2,3,0.9419,0.9495,0.2079,0.2455
3,4,0.9520,0.9697,0.1170,0.1839
4,5,0.9722,0.9571,0.0990,0.1568
5,6,0.9646,0.9697,0.0740,0.1192
6,7,0.9596,0.9621,0.0620,0.0879
7,8,0.9773,0.9722,0.0461,0.0850


C:\Users\SukhPc\AppData\Local\Temp\ipykernel_56124\1514501321.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\SukhPc\AppData\Local\Temp\ipykernel_56124\1514501321.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Notebook Summary

This deep learning task uses a CNN to classify 8 by 8 handwritten digit images from a public non-synthetic benchmark dataset. The baseline CNN was compared with exactly one experimental configuration that added dropout before the classifier layer. Both models trained successfully and generated test metrics and plots. The dropout model offers a simple regularization comparison, but the dataset is small and much easier than real production image tasks. Future work should validate on larger images and more varied acquisition settings.

In [6]:
metrics = {
    'rows': int(df.shape[0]),
    'features': int(df.shape[1] - 1),
    'baseline_accuracy': float(baseline_history[-1]['accuracy']),
    'experimental_accuracy': float(experimental_history[-1]['accuracy']),
    'baseline_loss': float(baseline_history[-1]['loss']),
    'experimental_loss': float(experimental_history[-1]['loss']),
    'epochs': 8,
    'experiment': 'Added dropout=0.25 before final linear layer'
}
with open(OUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
metrics


{'rows': 1797,
 'features': 64,
 'baseline_accuracy': 0.9772727272727273,
 'experimental_accuracy': 0.9722222222222222,
 'baseline_loss': 0.04607984271239151,
 'experimental_loss': 0.08498384744267572,
 'epochs': 8,
 'experiment': 'Added dropout=0.25 before final linear layer'}